# Support Vector Machine (SVM)

## Importing the libraries

In [31]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Importing the dataset

In [32]:
dataset = pd.read_csv('Employee_Performance_Retention.csv')
# Drop the 'Employee_ID' and 'Unnamed: 10' columns as they contain missing values or are not needed
dataset = dataset.drop(['Employee_ID', 'Unnamed: 10'], axis=1)
# Separate features and target
X = dataset.drop('Attrition', axis=1)
y = dataset['Attrition']
categorical_cols = X.select_dtypes(include=['object', 'category']).columns
numerical_cols = X.select_dtypes(include=np.number).columns
print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Categorical columns: Index(['Department', 'Job_Satisfaction_Level', 'Promotion_in_Last_2_Years'], dtype='object')
Numerical columns: Index(['Age', 'Years_of_Experience', 'Monthly_Working_Hours',
       'Training_Hours_per_Year', 'Performance_Rating'],
      dtype='object')


## Splitting the dataset into the Training set and Test set

In [33]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

# Separate features and target
X = dataset.drop(['Attrition'], axis=1)
y = dataset['Attrition']

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns
numerical_cols = X.select_dtypes(include=np.number).columns

# Apply One-Hot Encoding to categorical features
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)], remainder='passthrough')
X = ct.fit_transform(X)

# Encode the target variable
le = LabelEncoder()
y = le.fit_transform(y)

# Split the dataset into the Training set and Test set with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0, stratify=y)

In [34]:
print(X_train)

[[  0.   0.   0. ... 195.  44.   3.]
 [  0.   0.   0. ... 201.  27.   3.]
 [  1.   0.   0. ... 172.  27.   3.]
 ...
 [  0.   0.   0. ... 182.   9.   4.]
 [  0.   0.   0. ... 197.  33.   5.]
 [  0.   0.   0. ... 180.  40.   1.]]


In [35]:
print(y_train)

[1 1 0 ... 0 0 1]


In [36]:
print(X_test)

[[  0.   0.   0. ... 154.  23.   2.]
 [  0.   1.   0. ... 208.  49.   1.]
 [  0.   0.   0. ... 148.  21.   1.]
 ...
 [  0.   1.   0. ... 218.   9.   1.]
 [  1.   0.   0. ... 145.   5.   1.]
 [  1.   0.   0. ... 231.  37.   3.]]


In [37]:
print(y_test)

[0 1 0 ... 0 0 0]


## Feature Scaling

In [38]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Training the SVM model on the Training set

In [39]:
from sklearn.svm import SVC
classifier_linear = SVC(kernel = 'linear', random_state = 0, class_weight='balanced')
classifier_linear.fit(X_train, y_train)

SVC(class_weight='balanced', kernel='linear', random_state=0)

## Saving the model

In [40]:
import pickle

# Save the trained linear SVM model to a file
filename = 'svm_linear_model.pkl'
pickle.dump(classifier_linear, open(filename, 'wb'))

print(f"Linear SVM model saved to {filename}")

Linear SVM model saved to svm_linear_model.pkl


In [41]:
from sklearn.svm import SVC
classifier_rbf = SVC(kernel = 'rbf', random_state = 0, class_weight='balanced')
classifier_rbf.fit(X_train, y_train)

SVC(class_weight='balanced', random_state=0)

In [42]:
from sklearn.svm import SVC
classifier_poly = SVC(kernel = 'poly', random_state = 0, class_weight='balanced')
classifier_poly.fit(X_train, y_train)

SVC(class_weight='balanced', kernel='poly', random_state=0)

## Predicting the Test set results

In [43]:
y_pred_linear = classifier_linear.predict(X_test)
print(np.concatenate((y_pred_linear.reshape(len(y_pred_linear),1), y_test.reshape(len(y_test),1)),1))

[[0 0]
 [0 1]
 [1 0]
 ...
 [0 0]
 [1 0]
 [1 0]]


In [44]:
y_pred_rbf = classifier_rbf.predict(X_test)
print(np.concatenate((y_pred_rbf.reshape(len(y_pred_rbf),1), y_test.reshape(len(y_test),1)),1))

[[1 0]
 [0 1]
 [0 0]
 ...
 [0 0]
 [1 0]
 [1 0]]


In [45]:
y_pred_poly = classifier_poly.predict(X_test)
print(np.concatenate((y_pred_poly.reshape(len(y_pred_poly),1), y_test.reshape(len(y_test),1)),1))

[[0 0]
 [1 1]
 [1 0]
 ...
 [1 0]
 [1 0]
 [1 0]]


## Making the Confusion Matrix

In [49]:
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np

# Get decision function scores for each model
decision_scores_linear = classifier_linear.decision_function(X_test)
decision_scores_rbf = classifier_rbf.decision_function(X_test)
decision_scores_poly = classifier_poly.decision_function(X_test)

# Apply the threshold of 0.8 to the decision function scores
threshold = 0.8
y_pred_linear_thresh = (decision_scores_linear > threshold).astype(int)
y_pred_rbf_thresh = (decision_scores_rbf > threshold).astype(int)
y_pred_poly_thresh = (decision_scores_poly > threshold).astype(int)

# Calculate confusion matrices and accuracy scores with the new thresholded predictions
cml_thresh = confusion_matrix(y_test, y_pred_linear_thresh)
cmr_thresh = confusion_matrix(y_test, y_pred_rbf_thresh)
cmp_thresh = confusion_matrix(y_test, y_pred_poly_thresh)

print(f"Confusion Matrix (Linear Kernel) with threshold {threshold}:\n", cml_thresh)
print(f"\nConfusion Matrix (RBF Kernel) with threshold {threshold}:\n", cmr_thresh)
print(f"\nConfusion Matrix (Polynomial Kernel) with threshold {threshold}:\n", cmp_thresh)

print(f"\nAccuracy (Linear Kernel) with threshold {threshold}: {accuracy_score(y_test, y_pred_linear_thresh)}")
print(f"Accuracy (RBF Kernel) with threshold {threshold}: {accuracy_score(y_test, y_pred_rbf_thresh)}")
print(f"Accuracy (Polynomial Kernel) with threshold {threshold}: {accuracy_score(y_test, y_pred_poly_thresh)}")

Confusion Matrix (Linear Kernel) with threshold 0.8:
 [[ 754 1058]
 [ 192  246]]

Confusion Matrix (RBF Kernel) with threshold 0.8:
 [[1581  231]
 [ 390   48]]

Confusion Matrix (Polynomial Kernel) with threshold 0.8:
 [[1601  211]
 [ 380   58]]

Accuracy (Linear Kernel) with threshold 0.8: 0.4444444444444444
Accuracy (RBF Kernel) with threshold 0.8: 0.724
Accuracy (Polynomial Kernel) with threshold 0.8: 0.7373333333333333
